## 2 Global Navigation

Authored by: Jiri Blaha

### [2A] Module Overview

The global navigation module transforms the computer vision output into a collision-free reference trajectory. It was designed with modularity in mind, so that it is importable from anywhere. The module is spread accross multiple files for better organization and separation of concerns. The specific directory structure is shown below.

>```txt
>classes/gp/
>├── __init__.py     # init
>├── planner.py      # planning
>├── plotter.py      # plotting
>└── utils.py        # utilities
>```

The main API used for the project are the `GlobalPlanner` and `Plotter` classes. The former houses the general planning functionality, while the latter provides useful plots: either of generated paths, or comparison plots. Import of the aforementioned classes is, as already mentioned, accessible from anywhere within the project using the standard import syntax of the Python language:

In [1]:
# Module import (API used for the project)
from classes.gp import GlobalPlanner, Plotter

which exposes all the necessary functionality of global planning (and related plotting). This functionality is provided byt the `.plan()` method, owned by `GlobalPlanner`, and the `.path()` method, owned by `Plotter`. For more details, see the [here](./classes/gp/planner.py) and [here](./classes/gp/plotter.py).

### [2B] Independent Showcase

To show the independent functionality of the module, we provide dummy computer vision data below. Generally, the computer vision output (and thereby global planning input) obeys the following structure:

>```python
>cv_data = [
>    [W, H],                                     # map size (width, height)
>    np.array([[obs0_v0], ..., [obs0_vn]]),      # vertices of obstacle 0
>    ...,                                        # vertices of other obs.
>    np.array([[obsM_v0], ..., [obsM_vk]]),      # vertices of obstacle M
>    np.array([s_x, s_y]),                       # start coordinates (x,y)
>    np.array([g_x, g_y])                        # goal coordinates (x,y)
>]
>```

which can be changed to obtain different result, yet we need to be aware of edge cases where for some maps, no path can be found. This issue concerns only unreasonably designed maps, *i.e.*, those with very tight gaps, bad obstacle spacing, etc. All coordinates are in the computer vision frame. Using this information, the planner computes a reference path based on a visibility graph and an A* search.

In [2]:
import numpy as np

# Load "CV" data
data = [
        [260.0, 200.0],
        np.array([[60, 40], [90, 40], [90, 80], [60, 80]]),
        np.array([[130, 40], [200, 40], [200, 70], [130, 70]]),
        np.array([[120, 120], [160, 120], [160, 170], [120, 170]]),
        np.array([[40, 130], [70, 140], [65, 160], [35, 170], [25, 150]]),
        np.array([[200, 110], [230, 120], [220, 145]]),
        np.array([[180, 10], [230, 15], [225, 35], [175, 30]]),
        np.array([20, 20]),
        np.array([240, 180]),
    ]

Next, let us explore the aforementioned API. First, we need to initialize an instance of both classes. Besides the required `data` argument, the planner can also take additional arguments, namely the Thymio's radius and the spacing of intermediate goals along the reference path. We provide sample values below but note these *do not* represent the final values chosen. The final values can be found [here](./constants.py).

In [ ]:
RADIUS = 2.0    # "Thymio radius" (circumscribed circle)
SPACING = 5.0   # Intermediate goals spacing

# Create GlobalPlanner instance
planner = GlobalPlanner(
    data,
    radius=RADIUS,
    spacing=SPACING
)

# Create Plotter instance
plotter = Plotter(planner)

# Plan and plot path
segments = planner.plan(data)   # plan
plotter.path()                  # plot

Notice that the instance creation of `plotter` requires an instance of `planner`, as can again be seen [here](./classes/gp/plotter.py). With instances created, a reference path can be planned and plotted using the methods mentioned above. It is best to showcase the planning module through a series of plots which best highlight the stages it goes through. Note that these plots were generated on the sample data above and not the actual data. In the figure below, the initial stage produced is the original, plain course without any annotations from `planner`. Next, visibility is introduced (see middle figure), after which we finally generate our reference path using the A* algorithm.

![](./images/planning_stages.svg)

The `GlobalPlanner` class (and its plotting counterpart) naturally extend to real Thymio runs. Besides the already mentioned API, we make use of the architecture in real runs in two other ways. Specifically, the `GlobalPlanner` and `Plotter` allow for:

* Moving the start and replanning from only this information. This functionality is achieved through the `start: Point = None` argument found in the `.plan()` method of `GlobalPlanner`. Specifically, we use this form of planning during kidnapping. We assume the environment Thymio navigates, *i.e.*, the obstacles and goal, stays constant in this state, except for the start.

* Overlaying the EKF estimate, if available, on top of the planned path to see the difference between planning and actual Thymio execution. This is available through the `.paths()` method of the `Plotter` class.

Below we provide examples outputs. The left figure indicates the moving start and is shown for the sample computer vision data. On the other hand, the right figure shows the overlay capability and was captured during an actual run in the testing environment.

![](./images/move_start_and_exec.svg)

### [2C] Deep-Dive & Explanation

This section intends to go deeper into the design considerations and ultimate choices that let the planner become what it is. Further, a concise explanation through mathematical rigor connected to not only the libraries used but also the course material itself is made. 

#### [2C-1] Design and Library Choices

The underpinning problem of global navigation is simple. We need to find a shortest reference path around polygonal obstacles for the robot to follow. On one hand, if we consider path planning as merely a pre-processing step, performance is not a main criterion. On the other, when accounting for the kidnapping case, we'd like to find a reference path in the least amount of time in order for us to resume motion without massive delays. Additionally, the question of what do deploy a path finding algorithm on arises as well, among others.

##### Why Visibility Graphs?

As can be inferred from the figures provided, or even from the code, the ultimate choice landed on visibility graphs, yet other canonical approaches were considered as well. Specifically, the following approaches were considered:

* **Occupancy grids**, which provide a uniform and simple structure but lose geometric precision and depend on resolution.

* **Visibility graphs**, which require careful geometric handling but are deterministic and able to provide the exact shortest path.

These consideration made visibility graphs a better choice. To manifest the choice even more, visibility graphs were also chosen since they pair well with robust computational geometry libraries (see later), and, most importantly, they fit our design and objectives best.

##### Why A* over Dijkstra?

The main issue with Dijkstra is that it searches exhaustively with cost only, while the A* algorithm adds a heuristic. As we navigate Euclidean space, the $\ell^2$-norm heuristic was chosen (see later) because it is admissible and consistent. Furthermore, A* search outperforms Dijkstra on visibility graphs with many vertices, which could dramatically reduce time for extreme maps. 

##### Why Inflate Obstacles?

Again, one can infer from the figures (or code) that the initial polygons, provided by computer vision, are being inflated prior to any visibility graph or A* computation. This inflation nothing but a Minkowski sum of the Thymio's radius and the original obstacles. The specific mathematics will be discussed later on, yet from a qualitative standpoint, this is merely an obvious safety measure.

##### Why Shapely/NetworkX?

The [`networkx`](https://networkx.org/en/) and [`shapely`](https://shapely.readthedocs.io/en/stable/)/[`shapely.geometry`](https://shapely.readthedocs.io/en/stable/geometry.html) libraries were chosen mainly for their easy setup, intuitive API, and general functionality they bring to this specific problem. As a nice-to-have, both also provide good documentation. Additionally, both were investigated at the very start of the project and they have proven to work. Planning also uses [`numpy`](https://numpy.org/doc/stable/) and [`matplotlib`](https://matplotlib.org/stable/users/index).

#### [2C-2] Conceptual Pipeline

Before diving into mathematics, and with the design choices in mind, the author takes the liberty to outline the high-level flow of information, *i.e.*, to showcase the conceptual pipeline of the planning module. This pipeline has a structure of the form:

1. The computer vision module produces `data` as seen in above code.
2. `GlobalPlanner` inflates polygonal obstacles and creates vertices.
3. `VisibilityGraph` determines valid connections between vertices.
4. `networkx` runs an A* search over the graph from the previous step.
5. Spacing-dependent segments get generated from the previous result.
6. `Plotter` provides visual validation of the found path and segments.

Looking at the broader scope of the project, it is important to note that step 6 is entirely optional and does not enhance any core functinoality. Essentially, the pipeline could end at step 5, when triplets `(x,y,theta)` are passed to the motion control module.

#### [2C-3] Rigorous Background

The qualitative reasoning we have developed thus far lets us explore the problem of planning from a more rigorous perspective. Unfortunately, to keep the report within reasonable limits, only the main results we have used are shown in the following text.

##### Geometry and Constraints

Let the map domain be

$$
\Omega = [0,W]\times[0,H]\subset\mathbb{R}^2
$$

where $W$ and $H$ are the width and height of the map, respectively. Additionally, we project the start and goal points, provided by computer vision, into this domain. Given some point $p=(x,y)\in\mathbb{R}^2$, its projection to $\Omega$ is implemented as

$$
x'=\min\{\max\{x,0\}, W\}{,}\quad y'=\min\{\max\{y,0\}, H\}{,}\quad(x',y')\in\Omega
$$

to make sure the planner never reasons about configurations outside the map. For code implementation, see `clamp_point` [here](./classes/gp/planner.py). Furthermore, let the obstacle set provided by the computer vision module be

$$
\mathcal{O} = \{\mathcal{O}_1,\dots,\mathcal{O}_M\}{,}\quad\mathcal{O}_i = \{v_{i_1},\dots,v_{i_{k_i}}\}
$$

where each $\mathcal{O}_i$ is a simple polygon. These polygons partition the map into

$$
\Omega_\mathrm{av} = \Omega\setminus\bigcup\limits_{i=1}^m\mathcal{O}_i\subset\Omega\quad(\subseteq\Omega\ \mathrm{if}\ \mathcal{O}=\emptyset)
$$

wherein we refer to $\Omega_\mathrm{av}$ as available or free space, *i.e.*, what is left of the map for Thymio to navigate.

##### Obstacle Inflation

As discussed before, to create a safety margin for the robot, each polygon is inflated by some $r\geq0$. This is equivalent to

$$
O_i^{(r)} = \mathcal{O}_i\oplus\mathbb{B}_r^{(0)}{,}\quad\mathbb{B}_r^{(0)}=\{x\in\mathbb{R}^2:\lVert x\rVert\leq r\}
$$

where $\mathbb{B}_r^{(0)}$ is a closed disk and the $\oplus$ operator denotes the [Mikowski sum](https://en.wikipedia.org/wiki/Minkowski_addition). The `shapely` library we used computes this via the `.buffer()` method of its `Polygon` class (see [here](./classes/gp/planner.py)), which is nothing but an approximation of the Minkowski sum. Providing `join_style=2` as one of the arguments makes sure to preserve sharp corners of the original polygons. Naturally, the newely constructed shapes provide vertices of the visibility graph after being clamped to the map domain $\Omega$.

##### Visibility Graph

The visibility graph lies at the core of global planning. Let

$$
V = \{v_1,\dots,v_N\}
$$

denote the set of all vertices of all inflated polygons, and eventually the start and goal points. Two arbitrary vertices $a,b\in V$ are connected by an edge if and only if the closed segment (`shapely`'s `LineString`)

$$
[a,b] = \{a+t(b-a):t\in[0,1]\}
$$

satisfies the condition

$$
[a,b]\subseteq\Omega_\mathrm{av}
$$

*i.e.*, lies entirely within the available space. This verification is done using the `.within()` and `.crosses()` methods of the aforementioned `LineString` object (see [here](./classes/gp/graph.py)). Additionally, we allow intersection along the inflated obstacles' boundaries. Subsequently, the visibility graph is stored using `networkx` as a `nx.Graph` object. Formally, it is

$$
G=(V,E){,}\quad E=\{(a,b):a,b\in V, [a,b]\subseteq\Omega_\mathrm{av}\}
$$

with each edge receiving the weight

$$
w(a,b)=\lVert b-a\rVert_2
$$
*i.e.*, a standard Euclidean weight mentioned before.

##### A* Path Search

Let $s,g\in V$ denote the start and goal, respectively. Additonally, let $\mathcal{P}(s,g)$ denote the set of all simple paths in $G=(E,V)$, connecting $s$ to $g$. For any such path $p=(v_0=s,\dots,v_k=g)$, its cost is

$$
g(p) = \sum\limits_{i=0}^{k-1}w(v_i,v_{i+1})
$$

and the objective of global planning is therefore

$$
p^\star = \argmin_{p\in\mathcal{P}(s,g)}g(p)
$$

essentially guiding Dijkstra with an admissible heuristic. For every vertex $x\in V$, define

$$
f(x)=g(x)+h(x)
$$

where $g(x)$ is the cost of the best known path from $s$ to $x$ and $h(x)$ is an estimate of the remaining cost to reach $g$. We choose

$$
h(x) = \lVert x-g\rVert_2
$$
*i.e.*, again, an Euclidean heuristic. As seen in the course, A* expands vertices in order of *lowest* $f(x)$, *i.e.*,

$$
x_\mathrm{next}=\argmin_{x\in\mathcal{X}}f(x)
$$

where $\mathcal{X}$ is the set of still open vertices the path has not reached yet. Notice Dijkstra is recovered from A* by setting $h=0$. All aforementioned mathematics of this subsection is provided by the `.astar_path()` function of `networkx` (see [here](./classes/gp/planner.py)).

##### Segmentation

The A* result is a polyline 

$$
p^\star=(v_0,\dots,v_k){,}\quad v_i=(x_i,y_i)
$$

and in order for us to obtain waypoints, each polyline edge is sampled with $d>0$. For each consecutive pair $v_i, v_{i+1}$, define

$$
\xi = x_{i+1}-x_i{,}\quad\eta = y_{i+1} - y_i{,}\quad\ell_i=\sqrt{\xi^2+\eta^2}{,}
$$

so that samples along the segment are generated for 

$$
t = \frac{kd}{\ell_i}{,}\quad k=1,\dots,\left\lfloor\frac{\ell_i}{d}\right\rfloor
$$

where $\lfloor\cdot\rfloor$ denotes the floor operator. This ensures that an element of $p^\star$ is always the last point in the sequence. We now have

$$
x(t) = x_i+t\xi{,}\quad y(t)=y_i+t\eta{,}\quad\theta_i=\mathrm{atan2}(\eta,\xi)
$$

which we can return as `segments` (see [here](./classes/gp/planner.py) or [here](./main.py)). At a vertex from $p^\star$, we return an average of the previous and next $\theta$.

##### Validation and Plotting

Before running the Thymio, a plot of the reference path is shown (see `plotter.path()` [here](./main.py)) in order to validate the path shape. Because the `Plotter` class is mostly a visualization/utililty tool, the code is self-explanatory and uninteresting, and because we've covered it already in some detail, the author takes the liberty to omit a specific deep-dive and instead refers the reader [here](./classes/gp/plotter.py).